# Running Analysis
Reads Coros `.fit` files, filters to running activities only, and computes personal records.

## 0 · Dependencies

In [ ]:
# Install if needed (comment out after first run)
# %pip install fitparse pandas numpy

In [9]:
import glob
import warnings

import numpy as np
import pandas as pd
from fitparse import FitFile

from run_analyzer import RunAnalyzer, RUNNING_SPORT_TYPES

warnings.filterwarnings("ignore")

## 1 · Discover files

In [10]:
files = glob.glob("../data/coros/*.fit")
print(f"Detected {len(files)} FIT files")

Detected 462 FIT files


## 2 · Parse FIT files → DataFrames

Reads every `record` message (GPS/sensor data rows) and the `sport` / `session` messages to identify activity type.  
Non-running activities (cycling, swimming, strength, …) are silently skipped.

In [11]:
# Fields we want from each record message
RECORD_FIELDS = [
    "timestamp",
    "distance",
    "heart_rate",
    "enhanced_speed",
    "speed",
    "cadence",
    "power",
    "accumulated_power",
    "enhanced_altitude",
    "altitude",
    "step_length",
    "vertical_oscillation",
    "stance_time",
    "position_lat",
    "position_long",
]


def _get_sport(fit: FitFile) -> str | None:
    """
    Extract activity sport string from a FIT file.
    Checks 'sport' messages first, falls back to 'session'.
    Returns the lowercase sport string or None.
    """
    for msg_type in ("sport", "session"):
        for msg in fit.get_messages(msg_type):
            for field in msg:
                if field.name in ("sport", "sub_sport", "activity_type"):
                    if field.value is not None:
                        val = str(field.value).lower().replace(" ", "_")
                        return val
    return None


def parse_fit_file(path: str) -> tuple[pd.DataFrame | None, str | None]:
    """
    Parse a single FIT file.

    Returns
    -------
    (DataFrame, sport_string)  if it's a running activity
    (None, sport_string)       if it's a different sport
    (None, None)               if the file is corrupt / unreadable
    """
    try:
        fit = FitFile(path)
        sport = _get_sport(fit)

        # Skip non-running activities
        if sport is not None and sport not in RUNNING_SPORT_TYPES:
            return None, sport

        rows = []
        for msg in fit.get_messages("record"):
            row = {f: None for f in RECORD_FIELDS}
            for field in msg:
                if field.name in RECORD_FIELDS:
                    row[field.name] = field.value
            rows.append(row)

        if not rows:
            return None, sport

        df = pd.DataFrame(rows)

        # Semicircle → degrees for lat/long
        for col in ("position_lat", "position_long"):
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce") * (180 / 2**31)

        return df, sport

    except Exception as exc:
        print(f"  [WARN] Could not parse {path}: {exc}")
        return None, None


# ---------------------------------------------------------------------------
# Load all files
# ---------------------------------------------------------------------------
runs: list[pd.DataFrame] = []
run_paths: list[str] = []
skipped: dict[str, int] = {}

for path in sorted(files):
    df, sport = parse_fit_file(path)
    if df is not None:
        runs.append(df)
        run_paths.append(path)
    else:
        label = sport or "unknown/corrupt"
        skipped[label] = skipped.get(label, 0) + 1

print(f"Loaded  : {len(runs)} running activities")
if skipped:
    print("Skipped :", ", ".join(f"{v}× {k}" for k, v in sorted(skipped.items())))

  [WARN] Could not parse ../data/coros/468187902835195904.fit: Invalid field size 1 for type 'uint32' (expected a multiple of 4)
  [WARN] Could not parse ../data/coros/468234812465905693.fit: Invalid field size 1 for type 'uint32' (expected a multiple of 4)
  [WARN] Could not parse ../data/coros/468279331949412361.fit: Invalid field size 1 for type 'uint32' (expected a multiple of 4)
  [WARN] Could not parse ../data/coros/468327544570019841.fit: Invalid field size 1 for type 'uint32' (expected a multiple of 4)
  [WARN] Could not parse ../data/coros/468350907044626438.fit: Invalid field size 1 for type 'uint32' (expected a multiple of 4)
  [WARN] Could not parse ../data/coros/468367676207562753.fit: Invalid field size 1 for type 'uint32' (expected a multiple of 4)
  [WARN] Could not parse ../data/coros/468374142278729735.fit: Invalid field size 1 for type 'uint32' (expected a multiple of 4)
  [WARN] Could not parse ../data/coros/468388967434190861.fit: Invalid field size 1 for type 'uin

## 3 · Quick sanity check on raw data

In [4]:
if runs:
    print("Columns in a sample run:")
    print(runs[0].dtypes)
    print()
    print("First few rows:")
    display(runs[0].head())

Columns in a sample run:
timestamp               datetime64[ns]
distance                       float64
heart_rate                      object
enhanced_speed                 float64
speed                          float64
cadence                        float64
power                           object
accumulated_power               object
enhanced_altitude              float64
altitude                        object
step_length                     object
vertical_oscillation            object
stance_time                     object
position_lat                   float64
position_long                  float64
dtype: object

First few rows:


,timestamp,distance,heart_rate,enhanced_speed,speed,cadence,power,accumulated_power,enhanced_altitude,altitude,step_length,vertical_oscillation,stance_time,position_lat,position_long
0,2025-02-08 12:09:12,NaN,None,NaN,NaN,0.0,None,None,NaN,None,None,None,None,NaN,NaN
1,2025-02-08 12:09:14,NaN,None,NaN,NaN,0.0,None,None,NaN,None,None,None,None,NaN,NaN
2,2025-02-08 12:09:16,NaN,None,NaN,NaN,NaN,None,None,NaN,None,None,None,None,NaN,NaN
3,2025-02-08 12:09:17,NaN,None,2.681,2.681,NaN,None,None,138.0,None,None,None,None,41.493594,2.136226
4,2025-02-08 12:09:17,0.0,None,NaN,NaN,NaN,None,None,NaN,None,None,None,None,NaN,NaN


## 4 · Build the analyzer

In [5]:
analyzer = RunAnalyzer(runs)
print(analyzer)

RunAnalyzer(26 runs, 0 flagged)


## 5 · All runs summary

In [6]:
summary = analyzer.summary()
display(summary)

,id,date,distance_km,time,avg_pace,avg_hr,avg_cadence,elev_gain_m,flagged
0,0,2025-02-08,2.56,12:32,4:53 /km,NaN,41.0,140.0,False
1,1,2025-01-12,2.56,11:03,4:19 /km,NaN,74.0,134.0,False
2,2,2024-12-28,16.75,1:55:15,6:53 /km,NaN,23.0,1294.0,False
3,3,2025-04-25,11.31,1:11:04,6:17 /km,161.0,83.0,155.0,False
4,4,2025-04-27,3.23,16:25,5:05 /km,159.0,89.0,22.0,False
5,5,2025-04-28,13.01,1:19:22,6:06 /km,153.0,80.0,170.0,False
6,6,2025-05-09,5.93,36:58,6:14 /km,158.0,82.0,100.0,False
7,7,2025-05-11,10.05,45:06,4:29 /km,176.0,88.0,20.0,False
8,8,2025-05-14,7.37,42:19,5:45 /km,158.0,82.0,91.0,False
9,9,2025-05-18,0.97,4:43,4:51 /km,155.0,89.0,12.0,False


## 6 · Flag runs with GPS anomalies

Use any of the methods below.  
Flagged runs are **excluded from all PR calculations** but remain in the summary table (shown as `flagged=True`).

In [7]:
# --- Option A: flag by run index ---
# analyzer.flag(0, 3)          # flag runs #0 and #3

# --- Option B: flag by date (flags ALL runs that day) ---
# analyzer.flag_by_date("2025-03-15")

# --- Unflag if you change your mind ---
# analyzer.unflag(3)

# --- See what's currently flagged ---
print("Flagged run IDs:", analyzer.flagged)
display(analyzer.summary()[analyzer.summary()["flagged"]])

Flagged run IDs: set()


,id,date,distance_km,time,avg_pace,avg_hr,avg_cadence,elev_gain_m,flagged


## 7 · Personal Records

In [8]:
display(analyzer.pr_table())

,distance,time,pace,date,run_id
0,1k,2:52,2:52 /km,2024-12-28,2.0
1,5k,22:17,4:27 /km,2025-05-11,7.0
2,10k,44:48,4:29 /km,2025-05-11,7.0
3,half,—,—,None,NaN
4,full,—,—,None,NaN


In [ ]:
# Drill into a specific distance
pr_5k = analyzer.best_for("5k")
if pr_5k:
    print(f"Best 5k : {pr_5k['time']}  ({pr_5k['pace']})  on {pr_5k['date']}  [run #{pr_5k['run_id']}]")
else:
    print("No 5k result yet.")

## 8 · Custom distances

Pass your own dict to `personal_records()` for any distance you care about.

In [ ]:
custom = analyzer.personal_records(distances={"3k": 3.0, "8k": 8.0, "15k": 15.0})
for label, pr in custom.items():
    if pr:
        print(f"{label:>5s}  {pr['time']:>10s}  {pr['pace']:>12s}  {pr['date']}")
    else:
        print(f"{label:>5s}  — (no qualifying run)")